# Reading and Writing Data with Spark

This notebook contains the code from the previous screencast. The only difference is that instead of reading in a dataset from a remote cluster, the data set is read in from a local file. You can see the file by clicking on the "jupyter" icon and opening the folder titled "data".

Run the code cell to see how everything works. 

First let's import SparkConf and SparkSession

In [1]:
%use spark

Since we're using Spark locally we already have both a sparkcontext and a sparksession running. We can update some of the parameters, such our application's name. Let's just call it "Our first Python Spark SQL example"

In [2]:
val jsc: JavaSparkContext = JavaSparkContext(sc)

Let's check if the change went through

In [3]:
jsc.conf.toDebugString()

spark.app.id=local-1581513298271
spark.app.name=Spark example
spark.driver.host=157e62db6cdf
spark.driver.port=43467
spark.executor.id=driver
spark.master=local
spark.repl.class.outputDir=/tmp/kotlin-jupyter4543394676199313387
spark.repl.class.uri=spark://157e62db6cdf:43467/classes

In [4]:
jsc.appName()

Spark example

As you can see the app name is exactly how we set it

Let's create our first dataframe from a fairly small sample data set. Througout the course we'll work with a log file data set that describes user interactions with a music streaming service. The records describe events such as logging in to the site, visiting a page, listening to the next song, seeing an ad.

In [5]:
val path = "data/sparkify_log_small.json"
val user_log = spark.read().json(path)

In [6]:
user_log.printSchema()

root
 |-- artist: string (nullable = true)
 |-- auth: string (nullable = true)
 |-- firstName: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- itemInSession: long (nullable = true)
 |-- lastName: string (nullable = true)
 |-- length: double (nullable = true)
 |-- level: string (nullable = true)
 |-- location: string (nullable = true)
 |-- method: string (nullable = true)
 |-- page: string (nullable = true)
 |-- registration: long (nullable = true)
 |-- sessionId: long (nullable = true)
 |-- song: string (nullable = true)
 |-- status: long (nullable = true)
 |-- ts: long (nullable = true)
 |-- userAgent: string (nullable = true)
 |-- userId: string (nullable = true)



In [7]:
fun Dataset<Row>.toHTML(limit: Int = 20, truncate: Int = 50): String {
    val sb = StringBuilder()
    sb.append("<html><body>")
    sb.append("<table><tr>")
    sb.append(schema().fieldNames().map { "<th style=\"text-align:left\">${it}</th>"}.joinToString(""))
    sb.append("</tr>")
    limit(limit).collectAsList().forEach { row ->
        sb.append("<tr>")
        (0 until row.size()).map {
            if (row[it] != null) row[it].toString() else ""
        }.forEach {
            val truncated = if (truncate > 0 && it.length > truncate) {
                if (truncate < 4) it.substring(0, truncate) else it.substring(0, truncate - 3) + "..."
            } else { it }
            sb.append("<td style=\"text-align:left\" title=\"$it\">$truncated</td>")
        }
        sb.append("</tr>")
    }
    sb.append("</table>")
    if(limit < count())
        sb.append("<p>... only showing top $limit rows</p>")
    sb.append("</body></html>")
    return sb.toString()
}

In [8]:
user_log.describe()

summary,artist,auth,firstName,gender,itemInSession,lastName,length,level,location,method,page,registration,sessionId,song,status,ts,userAgent,userId
count,8347,10000,9664,9664,10000,9664,8347,10000,9664,10000,10000,9664,10000,8347,10000,10000,9664,10000
mean,461.0,,,,19.6734,,249.6486587492503,,,,,1.5046953695887393E12,4436.7511,Infinity,202.8984,1.5137859954164E12,,1442.4413286423842
stddev,300.0,,,,25.382114916132597,,95.00437130781461,,,,,8.47314252131657E9,2043.1281541827557,NaN,18.041791154505876,3.2908288623601213E7,,829.8909432082613
min,!!!,Guest,Aakash,F,0,Acevedo,1.12281,free,"Aberdeen, WA",GET,About,1463503881284,9,#1,200,1513720872284,"""Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10) ...",
max,ÃÂlafur Arnalds,Logged Out,Zoie,M,163,Zuniga,1806.8371,paid,"Yuma, AZ",PUT,Upgrade,1513760702284,7144,wingless,404,1513848349284,Mozilla/5.0 (compatible; MSIE 9.0; Windows NT 6...,999


In [9]:
user_log.show(1)

+-------------+---------+---------+------+-------------+--------+---------+-----+--------------------+------+--------+-------------+---------+--------------------+------+-------------+--------------------+------+
|       artist|     auth|firstName|gender|itemInSession|lastName|   length|level|            location|method|    page| registration|sessionId|                song|status|           ts|           userAgent|userId|
+-------------+---------+---------+------+-------------+--------+---------+-----+--------------------+------+--------+-------------+---------+--------------------+------+-------------+--------------------+------+
|Showaddywaddy|Logged In|  Kenneth|     M|          112|Matthews|232.93342| paid|Charlotte-Concord...|   PUT|NextSong|1509380319284|     5132|Christmas Tears W...|   200|1513720872284|"Mozilla/5.0 (Win...|  1046|
+-------------+---------+---------+------+-------------+--------+---------+-----+--------------------+------+--------+-------------+---------+------

In [10]:
user_log.takeAsList(5)

[[Showaddywaddy,Logged In,Kenneth,M,112,Matthews,232.93342,paid,Charlotte-Concord-Gastonia, NC-SC,PUT,NextSong,1509380319284,5132,Christmas Tears Will Fall,200,1513720872284,"Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.125 Safari/537.36",1046], [Lily Allen,Logged In,Elizabeth,F,7,Chase,195.23873,free,Shreveport-Bossier City, LA,PUT,NextSong,1512718541284,5027,Cheryl Tweedy,200,1513720878284,"Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.143 Safari/537.36",1000], [Cobra Starship Featuring Leighton Meester,Logged In,Vera,F,6,Blackwell,196.20526,paid,Racine, WI,PUT,NextSong,1499855749284,5516,Good Girls Go Bad (Feat.Leighton Meester) (Album Version),200,1513720881284,"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_4) AppleWebKit/537.78.2 (KHTML, like Gecko) Version/7.0.6 Safari/537.78.2",2219], [Alex Smoke,Logged In,Sophee,F,8,Barker,405.99465,paid,San Luis Obispo-Paso Robles-Arroyo Grande, CA,PUT,NextSong,151300

In [11]:
val out_path = "data/sparkify_log_small.csv"

In [12]:
user_log.write().option("header", true).csv(out_path)

org.apache.spark.sql.AnalysisException: path file:/home/jovyan/work/data/sparkify_log_small.csv already exists.;
org.apache.spark.sql.execution.datasources.InsertIntoHadoopFsRelationCommand.run(InsertIntoHadoopFsRelationCommand.scala:114)
org.apache.spark.sql.execution.command.DataWritingCommandExec.sideEffectResult$lzycompute(commands.scala:104)
org.apache.spark.sql.execution.command.DataWritingCommandExec.sideEffectResult(commands.scala:102)
org.apache.spark.sql.execution.command.DataWritingCommandExec.doExecute(commands.scala:122)
org.apache.spark.sql.execution.SparkPlan$$anonfun$execute$1.apply(SparkPlan.scala:131)
org.apache.spark.sql.execution.SparkPlan$$anonfun$execute$1.apply(SparkPlan.scala:127)
org.apache.spark.sql.execution.SparkPlan$$anonfun$executeQuery$1.apply(SparkPlan.scala:155)
org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:151)
org.apache.spark.sql.execution.SparkPlan.executeQuery(SparkPlan.scala:152)
org.apache.spark.sql.execution.SparkPlan

In [13]:
val user_log_2 = spark.read().option("header", true).csv(out_path)

In [14]:
user_log_2.printSchema()

root
 |-- artist: string (nullable = true)
 |-- auth: string (nullable = true)
 |-- firstName: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- itemInSession: string (nullable = true)
 |-- lastName: string (nullable = true)
 |-- length: string (nullable = true)
 |-- level: string (nullable = true)
 |-- location: string (nullable = true)
 |-- method: string (nullable = true)
 |-- page: string (nullable = true)
 |-- registration: string (nullable = true)
 |-- sessionId: string (nullable = true)
 |-- song: string (nullable = true)
 |-- status: string (nullable = true)
 |-- ts: string (nullable = true)
 |-- userAgent: string (nullable = true)
 |-- userId: string (nullable = true)



In [15]:
user_log_2.takeAsList(2)

[[Showaddywaddy,Logged In,Kenneth,M,112,Matthews,232.93342,paid,Charlotte-Concord-Gastonia, NC-SC,PUT,NextSong,1509380319284,5132,Christmas Tears Will Fall,200,1513720872284,"Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.125 Safari/537.36",1046], [Lily Allen,Logged In,Elizabeth,F,7,Chase,195.23873,free,Shreveport-Bossier City, LA,PUT,NextSong,1512718541284,5027,Cheryl Tweedy,200,1513720878284,"Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.143 Safari/537.36",1000]]

In [16]:
user_log_2.select("userID").show()

+------+
|userID|
+------+
|  1046|
|  1000|
|  2219|
|  2373|
|  1747|
|  1747|
|  1162|
|  1061|
|   748|
|   597|
|  1806|
|   748|
|  1176|
|  2164|
|  2146|
|  2219|
|  1176|
|  2904|
|   597|
|   226|
+------+
only showing top 20 rows



In [18]:
user_log_2.takeAsList(1)

[[Showaddywaddy,Logged In,Kenneth,M,112,Matthews,232.93342,paid,Charlotte-Concord-Gastonia, NC-SC,PUT,NextSong,1509380319284,5132,Christmas Tears Will Fall,200,1513720872284,"Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.125 Safari/537.36",1046]]